# Issue #229 — divergence-aware M×3 data profile (BJ-UNIFIED-DIVERGENCE-DATA-PROFILE-001)

REWORK 2026-08-15 (REWORK_TYPE=method).  This notebook is a thin readable
orchestration layer: it runs the reusable `src/divergence_profile` package
top-to-bottom, renders every committed machine-readable table/figure and
verifies the immutable package.  All durable evidence lives in
`results/bj229_divergence_profile_v1/`.

## Setup

Run from the repository root with the project's pinned environment
(`PYTHONPATH=src .venv/bin/python`).  The package reads only the four
accepted upstream frozen packages (#203/#204/#214/#180), each
SHA256-verified before use; no settlement/result table is opened.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Notebook may be executed from the repo root or from its own directory.
ROOT = Path.cwd().resolve()
if not (ROOT / "results").exists():
    ROOT = ROOT.parents[2]
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from divergence_profile import builder, contract, manifest, sources, figures
from divergence_profile import probabilities as prob
from divergence_profile import profile as prof

PACKAGE = contract.package_path(ROOT)
print("root:", ROOT)
print("package:", PACKAGE)
print("pandas:", pd.__version__)

## Build the package

`builder.build` assembles the canonical cell frame on the #204 common-cell
axis with **match-level** #203 timestamps (kickoff, decision epoch,
per-feature observed-at), materializes the six representations, computes
per-cell divergence features, runs DQ-01..DQ-12 (fail closed) and writes the
immutable package atomically (staging + fsync + rename).

In [ ]:
receipt = builder.build(ROOT)
target = builder.write_package(receipt, force=True)
manifest.write_replay(target, receipt, ROOT)
manifest.write_sha256sums(target)
print("BUILT", target)
print("cells:", receipt.cells.shape)
print("disposition:", receipt.disposition)

## REQ-D evidence

One evidence row per executable acceptance requirement REQ-D01..D10.
Every `pass` value is computed from DQ receipts / causal availability / file
presence — never a hardcoded True.

In [ ]:
evidence = receipt.evidence
display(evidence[["requirement", "name", "pass"]])
failed = evidence.loc[~evidence["pass"].astype(bool), "requirement"].tolist()
print("FAILED:", failed if failed else "none")

## Census and identity completeness

Exact date/issue/match/cell census by cohort.  Matches are counted as
**(issue, match_idx)** pairs (the audited `match_idx.nunique()` undercounted
5,395 vs the exact 119,235).

In [ ]:
display(receipt.tables["census_by_cohort"])
display(receipt.tables["issue_count_distribution"])
display(receipt.tables["complete_matches_per_n"])

## Missingness and intersection coverage

Missingness per representation and BJ/EU/POWER/SP intersection coverage.

In [ ]:
display(receipt.tables["missingness_matrix"])
display(receipt.tables["intersection_coverage"])
display(receipt.tables["source_coverage_strata"])

## Distributions

Native-line, league, year and outcome-code distributions; Beijing SP and
normalized implied-probability distributions per representation.

In [ ]:
display(receipt.tables["line_league_year_distribution"])
display(receipt.tables["outcome_distribution"])
display(receipt.tables["sp_and_implied_distribution"])
for key in ("probability_distribution_qbj_calibrated",
            "probability_distribution_eu_bjline",
            "probability_distribution_power",
            "probability_distribution_sp_devig"):
    display(receipt.tables[key].head(20))

## Reconciliation and divergence

BJ raw-vs-calibrated reconciliation, EU-vs-BJ / POWER-vs-BJ per-outcome
scatter tables by native line, EU/POWER direction agreement matrix and
divergence magnitude tail.  Divergence values are per outcome cell (the
audited total-variation / max-component scalars are gone).

In [ ]:
display(receipt.tables["bj_raw_vs_calibrated_reconciliation"])
display(receipt.tables["eu_vs_bj_scatter_table"].head(20))
display(receipt.tables["power_vs_bj_scatter_table"].head(20))
display(receipt.tables["direction_agreement_matrix"])
display(receipt.tables["divergence_magnitude_tail"])

## Development-only calibration

Reliability/calibration on development labels (<=2023) and predicted-SP vs
settlement-SP diagnostics (REQ-D06/D08).

In [ ]:
display(receipt.tables["development_only_calibration"])
display(receipt.tables["predicted_sp_vs_settlement_sp"])

## Data-quality ledger

DQ-01..DQ-12 receipts and the stable exclusion ledger.  Checks run on the
exact match-level timestamps; DQ-12 scans ALL payloads (inference cells AND
development labels) for the 0.65 payout boundary.

In [ ]:
for name, frame in receipt.checks.items():
    print(f"{name}: {len(frame)} rows")
display(receipt.tables["exclusion_summary"])
display(receipt.tables["outlier_heavy_tail"])

## Chronological drift

Per-year support, calibrated availability and source mix.

In [ ]:
display(receipt.tables["drift_table"])
display(receipt.tables["supported_issues_per_year"])

## Figures

Every figure below is rendered from a committed machine-readable table.

In [ ]:
from IPython.display import Image as IPImage
import os

figures_dir = target / "figures"
for name in sorted(os.listdir(figures_dir)):
    if name.endswith(".png"):
        display(IPImage(filename=str(figures_dir / name)))

## Package verification

Fail-closed verification: every file under the package matches
SHA256SUMS.json, the handoff schema validates, and the REQ-D evidence
closure holds (no READY_FOR_209 with failed requirements).

In [ ]:
hashes = sources.verify_package_hashes(target)
print(f"VERIFIED {target}: {len(hashes)} files match SHA256SUMS.json")
import json
schema = json.loads((target / "handoff_schema.json").read_text())
print("handoff schema columns:", len(schema.get("columns", [])))
print("reviewed commit:", json.loads((target / "MANIFEST.json").read_text()).get("reviewed_commit_sha"))

## Readiness disposition

`READY_FOR_209`, `READY_WITH_EXPLICIT_LIMITATIONS` or `BLOCKED_DATA_DEFECT`
— chosen by consuming the DQ failures, never hardcoded.

In [ ]:
print("disposition:", receipt.disposition)